# Tutorial 17: Cross-Species Ortholog Embeddings

This tutorial demonstrates **multi-species support** in embpy.
We embed orthologous genes from human and mouse using DNA and
protein models, then compare the embedding spaces to see how
well different models capture cross-species conservation.

### What this tutorial covers

1. Initialize BioEmbedders for human and mouse
2. Define ~20 ortholog pairs (human/mouse)
3. Generate DNA embeddings with multi-species and species-specific models
4. Generate protein embeddings with ESM-2
5. PCA reduction to a common dimensionality
6. Comprehensive visualization with `embpy.pl`
7. Cross-species annotation comparison

### Key species-aware features

- `BioEmbedder(organism='mouse')` -- resolves mouse genes via Ensembl
- `GeneAnnotator(organism='mouse')` -- STRING PPI uses mouse taxon (10090)
- Human-only sources (GTEx, HPA, GWAS) are skipped gracefully for non-human
- Model warnings when using species-specific models on the wrong organism

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import torch
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import embpy.pl as epl

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.1)
%matplotlib inline

---
## 1. Initialize BioEmbedders for human and mouse

In [ ]:
from embpy.embedder import BioEmbedder

human_embedder = BioEmbedder(device='auto', organism='human')
mouse_embedder = BioEmbedder(device='auto', organism='mouse')

print(f'Human embedder: device={human_embedder.device}, organism={human_embedder.organism}')
print(f'Mouse embedder: device={mouse_embedder.device}, organism={mouse_embedder.organism}')

---
## 2. Define ortholog pairs

We select well-known genes with clear 1:1 orthologs between human and mouse.

In [ ]:
ORTHOLOGS = {
    'TP53':   'Trp53',
    'BRCA1':  'Brca1',
    'BRCA2':  'Brca2',
    'EGFR':   'Egfr',
    'MYC':    'Myc',
    'KRAS':   'Kras',
    'PTEN':   'Pten',
    'RB1':    'Rb1',
    'AKT1':   'Akt1',
    'MTOR':   'Mtor',
    'VEGFA':  'Vegfa',
    'TNF':    'Tnf',
    'IL6':    'Il6',
    'GAPDH':  'Gapdh',
    'ACTB':   'Actb',
    'CDH1':   'Cdh1',
    'NOTCH1': 'Notch1',
    'WNT1':   'Wnt1',
    'SOX2':   'Sox2',
    'NANOG':  'Nanog',
}

human_genes = list(ORTHOLOGS.keys())
mouse_genes = list(ORTHOLOGS.values())
gene_families = [g.upper() for g in human_genes]

print(f'{len(ORTHOLOGS)} ortholog pairs defined')

---
## 3. DNA embeddings

We use Nucleotide Transformer v2 (multi-species) to embed both
human and mouse genes, producing embeddings in the same latent space.

In [ ]:
DNA_MODEL = 'nt_v2_100m'

human_dna_embs = {}
for gene in human_genes:
    try:
        emb = human_embedder.embed_gene(gene, model=DNA_MODEL, pooling_strategy='mean')
        human_dna_embs[gene] = emb
        print(f'  Human {gene:8s}  dim={emb.shape[0]}')
    except Exception as e:
        print(f'  Human {gene:8s}  FAILED: {e}')

mouse_dna_embs = {}
for gene in mouse_genes:
    try:
        emb = mouse_embedder.embed_gene(gene, model=DNA_MODEL, pooling_strategy='mean')
        mouse_dna_embs[gene] = emb
        print(f'  Mouse {gene:8s}  dim={emb.shape[0]}')
    except Exception as e:
        print(f'  Mouse {gene:8s}  FAILED: {e}')

print(f'\nHuman: {len(human_dna_embs)}/{len(human_genes)}, Mouse: {len(mouse_dna_embs)}/{len(mouse_genes)}')

---
## 4. Protein embeddings

ESM-2 is trained on UniRef and works across all species.
The same model embeds both human and mouse protein sequences.

In [ ]:
PROT_MODEL = 'esm2_650M'

human_prot_embs = {}
for gene in human_genes:
    try:
        emb = human_embedder.embed_protein(gene, model=PROT_MODEL, pooling_strategy='mean')
        human_prot_embs[gene] = emb
        print(f'  Human {gene:8s}  dim={emb.shape[0]}')
    except Exception as e:
        print(f'  Human {gene:8s}  FAILED: {e}')

mouse_prot_embs = {}
for gene in mouse_genes:
    try:
        emb = mouse_embedder.embed_protein(gene, model=PROT_MODEL, pooling_strategy='mean')
        mouse_prot_embs[gene] = emb
        print(f'  Mouse {gene:8s}  dim={emb.shape[0]}')
    except Exception as e:
        print(f'  Mouse {gene:8s}  FAILED: {e}')

print(f'\nHuman: {len(human_prot_embs)}/{len(human_genes)}, Mouse: {len(mouse_prot_embs)}/{len(mouse_genes)}')

---
## 5. Build combined AnnData

We combine human and mouse embeddings into a single AnnData
with species, gene family, and gene name as metadata.

In [ ]:
rows = []
dna_vecs = []
prot_vecs = []

for h_gene, m_gene in ORTHOLOGS.items():
    h_dna = human_dna_embs.get(h_gene)
    h_prot = human_prot_embs.get(h_gene)
    m_dna = mouse_dna_embs.get(m_gene)
    m_prot = mouse_prot_embs.get(m_gene)

    if h_dna is not None and h_prot is not None:
        rows.append({'gene': h_gene, 'species': 'human', 'family': h_gene.upper()})
        dna_vecs.append(h_dna)
        prot_vecs.append(h_prot)
    if m_dna is not None and m_prot is not None:
        rows.append({'gene': m_gene, 'species': 'mouse', 'family': h_gene.upper()})
        dna_vecs.append(m_dna)
        prot_vecs.append(m_prot)

obs = pd.DataFrame(rows)
obs.index = pd.Index([f'{r["species"]}_{r["gene"]}' for r in rows])

adata = ad.AnnData(obs=obs)
adata.obsm['X_dna_nt'] = np.stack(dna_vecs).astype(np.float32)
adata.obsm['X_protein_esm'] = np.stack(prot_vecs).astype(np.float32)

print(adata)
print(f'\nobsm keys: {list(adata.obsm.keys())}')
for k in adata.obsm:
    print(f'  {k}: {adata.obsm[k].shape}')

---
## 6. PCA reduction

Reduce both embedding spaces to 50 PCs for fair comparison.

In [ ]:
from embpy.pp import reduce_embeddings

N_PCA = min(50, adata.n_obs - 1)

for key in ['X_dna_nt', 'X_protein_esm']:
    reduce_embeddings(adata, obsm_key=key, n_components=N_PCA, scale=True)
    pca_key = f'{key}_pca'
    params = adata.uns[f'{pca_key}_params']
    print(f'{key:20s} -> {pca_key}  {adata.obsm[pca_key].shape}  '
          f'({params["total_variance_explained"]:.1%} variance)')

---
## 7. Visualizations

### Cross-species similarity heatmap

Do orthologs have higher similarity than non-orthologs?

In [ ]:
for key, label in [('X_dna_nt_pca', 'DNA (NT v2)'), ('X_protein_esm_pca', 'Protein (ESM-2)')]:
    epl.plot_similarity_heatmap(
        adata=adata, obsm_key=key, metric='cosine',
        labels=list(adata.obs_names),
        title=f'Cosine similarity: {label}',
        figsize=(12, 10),
    )
    plt.show()

### Embedding clustermaps

Do orthologs cluster together across species?

In [ ]:
epl.embedding_clustermap(
    adata, obsm_key='X_dna_nt_pca', n_obs=None,
    title='DNA embeddings (NT v2) -- human + mouse',
)
plt.show()

epl.embedding_clustermap(
    adata, obsm_key='X_protein_esm_pca', n_obs=None,
    title='Protein embeddings (ESM-2) -- human + mouse',
)
plt.show()

### Cross-model agreement

How much do DNA and protein embedding spaces agree?

In [ ]:
epl.cross_model_similarity(
    adata,
    obsm_keys=['X_dna_nt_pca', 'X_protein_esm_pca'],
    method='cosine_correlation',
    labels=['DNA (NT v2)', 'Protein (ESM-2)'],
    title='DNA vs Protein embedding agreement',
)
plt.show()

epl.cross_embedding_correlation(
    adata,
    obsm_key_a='X_dna_nt_pca',
    obsm_key_b='X_protein_esm_pca',
    method='pearson',
)
plt.show()

### UMAP colored by species and gene family

In [ ]:
from embpy.tl.clustering import leiden
from embpy.tl.dimred import compute_umap

for key in ['X_dna_nt_pca', 'X_protein_esm_pca']:
    compute_umap(adata, obsm_key=key, n_neighbors=min(10, adata.n_obs - 1))
    leiden(adata, obsm_key=key, resolution=0.5,
           n_neighbors=min(10, adata.n_obs - 1),
           key_added=f'leiden_{key}')

In [ ]:
for key, label in [('X_dna_nt_pca', 'DNA (NT v2)'), ('X_protein_esm_pca', 'Protein (ESM-2)')]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epl.plot_embedding_space(
        adata, obsm_key=key, color='species',
        basis=f'X_umap_{key}', title=f'{label} -- by species', ax=axes[0],
    )
    epl.plot_embedding_space(
        adata, obsm_key=key, color=f'leiden_{key}',
        basis=f'X_umap_{key}', title=f'{label} -- Leiden clusters', ax=axes[1],
    )
    plt.tight_layout()
    plt.show()

### All embeddings grid

In [ ]:
epl.all_embeddings(
    adata,
    obsm_keys=['X_dna_nt_pca', 'X_protein_esm_pca'],
    color='species',
    ncols=2,
)
plt.show()

### Radar chart: cluster property profiles

In [ ]:
epl.radar_chart(
    adata,
    properties=[],
    group_by='species',
    title='Species comparison (placeholder -- add annotations below)',
) if False else print('Radar chart will be shown after annotation in section 8')

### Parallel coordinates

In [ ]:
epl.parallel_coordinates(
    adata, obsm_key='X_protein_esm_pca',
    obs_features=[],
    n_dims=10, color_by='species',
    title='Parallel coordinates: Protein (ESM-2) by species',
)
plt.show()

### Star coordinates

In [ ]:
epl.star_coordinates(
    adata, obsm_key='X_protein_esm_pca', n_dims=10,
    color_by='species',
    title='Star coordinates: Protein (ESM-2)',
)
plt.show()

epl.star_coordinates(
    adata, obsm_key='X_dna_nt_pca', n_dims=10,
    color_by='species',
    title='Star coordinates: DNA (NT v2)',
)
plt.show()

### Dendrogram

In [ ]:
epl.dendrogram(
    adata, obsm_key='X_protein_esm_pca',
    metric='cosine', linkage_method='average',
    title='Hierarchical clustering: Protein (ESM-2)',
)
plt.show()

### Perturbation ranking

Find the most similar genes to human TP53 -- does mouse Trp53 rank first?

In [ ]:
if 'human_TP53' in adata.obs_names:
    epl.plot_perturbation_ranking(
        adata=adata, query='human_TP53', obsm_key='X_protein_esm_pca',
        query_label='human TP53', top_k=10,
    )
    plt.show()

---
## 8. Cross-species annotation comparison

Annotate human and mouse genes with pathways and PPI.
Human-only sources (GTEx, HPA, GWAS) are automatically skipped for mouse.

In [ ]:
from embpy.resources.gene_annotator import GeneAnnotator

human_ann = GeneAnnotator(organism='human')
mouse_ann = GeneAnnotator(organism='mouse')

ann_results = []
for h_gene, m_gene in ORTHOLOGS.items():
    h_pathways = human_ann.get_pathways(h_gene)
    m_pathways = mouse_ann.get_pathways(m_gene)
    h_ppi = human_ann.get_protein_interactions(h_gene)
    m_ppi = mouse_ann.get_protein_interactions(m_gene)

    h_n_path = sum(len(v) for v in h_pathways.values()) if isinstance(h_pathways, dict) else 0
    m_n_path = sum(len(v) for v in m_pathways.values()) if isinstance(m_pathways, dict) else 0
    h_n_ppi = len(h_ppi) if isinstance(h_ppi, list) else 0
    m_n_ppi = len(m_ppi) if isinstance(m_ppi, list) else 0

    ann_results.append({'family': h_gene, 'human_pathways': h_n_path, 'mouse_pathways': m_n_path,
                        'human_ppi': h_n_ppi, 'mouse_ppi': m_n_ppi})
    print(f'{h_gene:8s}/{m_gene:8s}  pathways: {h_n_path}/{m_n_path}  PPI: {h_n_ppi}/{m_n_ppi}')

df_ann = pd.DataFrame(ann_results)
df_ann

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_ann['human_pathways'], df_ann['mouse_pathways'], s=60, alpha=0.7)
for _, row in df_ann.iterrows():
    axes[0].annotate(str(row['family']), (row['human_pathways'], row['mouse_pathways']), fontsize=7)
axes[0].set_xlabel('Human pathways')
axes[0].set_ylabel('Mouse pathways')
axes[0].set_title('Pathway count: Human vs Mouse orthologs')
lim = max(df_ann[['human_pathways', 'mouse_pathways']].max()) + 1
axes[0].plot([0, lim], [0, lim], 'r--', alpha=0.5)

axes[1].scatter(df_ann['human_ppi'], df_ann['mouse_ppi'], s=60, alpha=0.7)
for _, row in df_ann.iterrows():
    axes[1].annotate(str(row['family']), (row['human_ppi'], row['mouse_ppi']), fontsize=7)
axes[1].set_xlabel('Human PPI partners')
axes[1].set_ylabel('Mouse PPI partners')
axes[1].set_title('PPI partners: Human vs Mouse orthologs')
lim2 = max(df_ann[['human_ppi', 'mouse_ppi']].max()) + 1
axes[1].plot([0, lim2], [0, lim2], 'r--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## Summary

### Multi-species API

| Component | How to set species |
|---|---|
| `BioEmbedder` | `BioEmbedder(organism='mouse')` |
| `GeneResolver` | `GeneResolver(species='mouse')` |
| `ProteinResolver` | `ProteinResolver(organism='mouse')` |
| `GeneAnnotator` | `GeneAnnotator(organism='mouse')` |
| `ProteinAnnotator` | `ProteinAnnotator(organism='mouse')` |

### Model-species compatibility

| Model category | Human | Mouse | Multi-species |
|---|---|---|---|
| Enformer | Yes | No | No |
| Borzoi | Yes | Yes (mouse weights) | No |
| NT v2 | Yes | Yes | Yes (trained on multi-species) |
| NTv3 | Yes | Yes | Yes (128k+ species) |
| GENA-LM multi | Yes | Yes | Yes |
| HyenaDNA | Yes | Yes | Trained on human, generalizes |
| ESM-2 / ESM-C / ESM3 | Yes | Yes | Yes (UniRef) |
| ProtT5 | Yes | Yes | Yes (UniRef) |

### Annotation source availability

| Source | Human | Mouse | Other |
|---|---|---|---|
| MyGene (pathways) | Yes | Yes | Yes |
| STRING-DB (PPI) | Yes | Yes | Yes (taxon-aware) |
| DoRothEA (TF regulons) | Yes | Yes | Via decoupler |
| GTEx (expression) | Yes | No | No |
| HPA (localization) | Yes | No | No |
| Open Targets (disease) | Yes | No | No |
| GWAS Catalog | Yes | No | No |